# 🧪 Fine-Tuning Eficiente de um Modelo de Linguagem com LoRA (PEFT)

Este tutorial conduz você, passo a passo, pelo processo de **ajuste fino (fine-tuning)** de um modelo de linguagem causal (ex.: `distilgpt2`) utilizando **LoRA** (*Low-Rank Adaptation*), uma das técnicas de **PEFT** (*Parameter-Efficient Fine-Tuning*).  
Ao final, você compreenderá os fundamentos matemáticos por trás do método e saberá como adaptar um modelo grande com muito menos recursos computacionais.

**O que você vai aprender:**
- A diferença entre *full fine-tuning* e *PEFT*.
- A ideia central do LoRA (matrizes de baixo posto).
- Como preparar dados, configurar, treinar e testar um modelo com LoRA.
- Comparar a qualidade das respostas antes e depois do ajuste fino.

## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

Neste notebook, usaremos a biblioteca `peft` (Hugging Face) para aplicar LoRA ao `distilgpt2`.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias (descomente a linha caso ainda não estejam instaladas):

In [3]:
!pip install transformers datasets peft accelerate torch

In [8]:
!pip install tranformers

ERROR: Could not find a version that satisfies the requirement tranformers (from versions: none)
ERROR: No matching distribution found for tranformers


Importe os módulos que serão utilizados ao longo do processo:

In [4]:
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

/home/joaovitor/Documents/Faculdade/1°Semestre/Tópicos avançados em Inteligência Artificial A/Second Avaliation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🔢 3. Carregar e Preparar o Dataset

Utilizaremos um arquivo `dataset.jsonl` onde cada linha contém uma instrução (`instruction`) e a saída desejada (`output`).  
Vamos converter cada exemplo em uma única string no formato:
```
Instruction: <instrução>
Output: <saída>
```
e depois dividir o conjunto em treino (80%) e validação (20%).

In [7]:
def convert_to_hf_format(example):
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset("json", data_files="dataset_gerado_500.jsonl")

# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)

# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 85
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 22
    })
})


## 🤖 4. Carregar o Modelo Pré-Treinado e o Tokenizador

Vamos carregar o `distilgpt2` – uma versão menor e mais rápida do GPT-2, ideal para experimentação.  
Como o tokenizador original não define um `pad_token`, usaremos o `eos_token` no lugar.

In [8]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 2028.55it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo carregado: google/flan-t5-large


## 🧪 5. Inferência ANTES do Fine-Tuning

Antes de qualquer treinamento, vamos ver como o modelo base responde a uma pergunta que está no nosso dataset.  
Isso servirá como **linha de base** para compararmos com o modelo ajustado.

In [9]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "What is the purpose of the report?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: What is the purpose of the report?
Resposta base: to describe the state of the health service


> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado ao nosso domínio.

## ✂️ 6. Tokenização do Dataset

Transformamos os textos em sequências de tokens que o modelo pode processar.  
Usaremos `padding="max_length"` e `truncation=True` para garantir que todas as amostras tenham o mesmo comprimento (128 tokens).

In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 85
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 22
    })
})


## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É essencial quando se utiliza quantização (QLoRA), mas também é recomendada mesmo sem quantização para melhor gerenciamento de memória.

In [11]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## 🧩 8. Configurar e Injetar LoRA

Agora definimos a configuração do LoRA:
- **r**: posto das matrizes de adaptação (quanto maior, mais capacidade, porém mais parâmetros).
- **lora_alpha**: fator de escala $\alpha$ (a atualização será multiplicada por $\alpha / r$).
- **target_modules**: os módulos do transformer onde aplicaremos LoRA. No GPT-2, `c_attn` e `c_proj` são as projeções de atenção.
- **lora_dropout**: dropout aplicado às matrizes LoRA para regularização.
- **bias**: "none" significa que não treinamos os vieses.
- **task_type**: como é um modelo de linguagem causal, usamos `CAUSAL_LM`.

Em seguida, criamos o modelo PEFT com `get_peft_model`, que insere os adaptadores LoRA e **congela** o resto do modelo.

In [13]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q",
        "k",
        "v",
        "o"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 792,587,264 || trainable%: 1.1907


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 🧱 9. Data Collator para Modelagem Causal

O `DataCollatorForLanguageModeling` prepara os lotes para o treinamento de linguagem causal (sem *masked language modeling*).  
Ele automaticamente desloca os rótulos para que a tarefa seja prever o próximo token.

In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.  
Como nosso dataset é pequeno, usaremos 100 épocas e uma taxa de aprendizado relativamente alta (`1e-3`).  
O `eval_steps` controla a frequência da avaliação no conjunto de validação.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",          # diretório de saída
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

## 🏋️ 11. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [12]:
trainer.train()

/home/joaovitor/Documents/Faculdade/1°Semestre/Tópicos avançados em Inteligência Artificial A/Second Avaliation/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
100,0.004192,0.002034
200,0.001756,0.000547
220,0.001414,0.000508


/home/joaovitor/Documents/Faculdade/1°Semestre/Tópicos avançados em Inteligência Artificial A/Second Avaliation/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/joaovitor/Documents/Faculdade/1°Semestre/Tópicos avançados em Inteligência Artificial A/Second Avaliation/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=220, training_loss=0.024853469516065988, metrics={'train_runtime': 4341.6323, 'train_samples_per_second': 0.196, 'train_steps_per_second': 0.051, 'total_flos': 495923783270400.0, 'train_loss': 0.024853469516065988, 'epoch': 10.0})

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.

In [5]:
model.save_pretrained("lora_finetuned_model03")
tokenizer.save_pretrained("tokenizer")

NameError: name 'model' is not defined

## 💻 14. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.

In [17]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("lora_finetuned_model03")
finetuned_tokenizer = AutoTokenizer.from_pretrained("tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 8394.32it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|██████████| 576/576 [00:00<00:00, 10915.86it/s]


In [20]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: What is the purpose of the report?
Resposta ajustada: 


## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o modelo base não conhecia nosso domínio; sua resposta era genérica ou incoerente.
- **Depois do fine-tuning:** com apenas uma fração dos parâmetros treinados (via LoRA), o modelo aprendeu a estrutura desejada e gera respostas alinhadas com os exemplos fornecidos.

Esse é o poder do **PEFT**: adaptar grandes modelos de forma rápida, barata e com resultados surpreendentes.

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|----------|-----------|
| **Full fine-tuning** | Atualiza todos os pesos do modelo. |
| **PEFT** | Atualiza apenas um pequeno número de parâmetros novos. |
| **LoRA** | Decompõe a atualização $\Delta W$ em $B A$, com $r \ll \min(d,k)$. |
| **r** | Posto da decomposição – controla a capacidade da adaptação. |
| **$\alpha$** | Fator de escala que ajusta a intensidade da adaptação. |
| **Target modules** | Camadas onde os adaptadores LoRA são inseridos. |

Agora você pode experimentar com outros valores de `r`, `lora_alpha`, ou até mesmo outros modelos!